# 🗺️ Google Maps Platform API 종합 마스터 노트북
### — 전체 입력 파라미터(Input) & 출력 속성(Output) & 실무 활용 시나리오(Scenarios) 가이드

본 노트북은 **Google Maps Platform**의 모든 핵심 Web Service API 및 최신 **Places API (New / Modern v1)**의 모든 기능, 입력 옵션, 반환 속성, 실무 비즈니스 시나리오를 심층적으로 검증하고 테스트할 수 있도록 작성되었습니다.

---

### 📌 다루는 API 서비스 및 핵심 시나리오 맵
| 번호 | API 서비스명 | 연동 방식 | 주요 입력(Inputs) & 출력(Outputs) | 핵심 실무 적용 시나리오 |
| :---: | :--- | :---: | :--- | :--- |
| **01** | **Geocoding API** | SDK / REST | 주소, 컴포넌트 필터, Bounds ➡️ 좌표, 세부 행정구역 컴포넌트, 정밀도, 뷰포트 | 주소 표준화/정제, 국가/우편번호 제한 검색, 지도 줌 바운딩박스 계산 |
| **02** | **Reverse Geocoding API** | SDK / REST | 위경도 좌표, ResultType, LocationType ➡️ 계층별 표준 주소 및 행정구역 매핑 | 모바일 GPS 위치를 도로명 주소로 변환, 라이드헤일링 픽업지 주소 자동입력 |
| **03** | **Places API (New) - 검색** | REST v1 | TextQuery, Circle/Rect 반경, OpenNow, MinRating, Price, RankPreference ➡️ 장소 목록 | "영업중 + 평점 4.5 이상" 조건부 맛집/시설 검색, 거리순/인기순 정렬 |
| **04** | **Places API (New) - 상세 (`*`)** | REST v1 | Place ID, FieldMask (`*`) ➡️ 50+ 속성 (리뷰, 영업시간, 편의시설, 전기차 충전, 휠체어) | 매장 상세 페이지 구축, 무장애(배리어프리) 시설 안내, EV 충전소 검색 |
| **05** | **Places Autocomplete API** | REST v1 | Partial Input, SessionToken, LocationBias, Origin ➡️ 추천 검색어, 직선거리, PlaceID | 검색창 실시간 자동완성, 세션 토큰을 통한 과금 최적화, 내 위치 기준 거리 표시 |
| **06** | **Directions API** | SDK / REST | Origin, Dest, Mode, Waypoints(`optimize:true`), TrafficModel, TransitOptions ➡️ 턴바이턴 경로, 소요시간 | 배송 차량 다중 경유지 최적 순서 계산(TSP), 실시간 교통 반영 도착 예정시간(ETA) |
| **07** | **Distance Matrix API** | SDK / REST | Origins[], Destinations[], Mode, DepartureTime, TrafficModel ➡️ N x M 거리 및 소요시간 | 라이드헤일링/배달 기사 최적 배차, 물류 거점-배송지 간 이동 비용 매트릭스 |
| **08** | **Elevation API** | SDK / REST | Locations[], Path[], Samples ➡️ 해발 고도, 측정 해상도, 고도 변화 프로파일 | 등산/사이클 경로 경사도 분석, 드론/UAV 비행 고도 지형 안전고도 확인 |
| **09** | **Time Zone API** | SDK / REST | Location, Timestamp ➡️ TimeZone ID, 표준 UTC 오프셋, 서머타임(DST) 오프셋 | 글로벌 항공/호텔 현지 시간 변환, IoT 기기 GPS 기반 시계 자동 동기화 |
| **10** | **Geolocation API** | SDK / REST | ConsiderIP, CellTowers[], WiFiAccessPoints[] ➡️ 추정 좌표, 오차 반경 | GPS 음영지역(실내/지하) 기기 위치 추정, Wi-Fi BSSID 기반 자산 트래킹 |
| **11** | **Roads API** | SDK / REST | Path[], Interpolate, Points[] ➡️ 도로 스냅 좌표, Place ID | 차량 GPS 궤적 보정(Snap to Roads), 주행 속도 제한(Speed Limit) 준수 모니터링 |

---


## 📦 0. 환경 설정 및 패키지 설치 가이드

본 프로젝트는 초고속 패키지 관리자 `uv` 및 표준 `pip` 환경을 모두 지원합니다.

### ⚡ Option A. `uv` 사용 (권장)
```bash
# 가상환경 생성 및 의존성 설치
uv sync

# Jupyter 커널 등록
uv run python -m ipykernel install --user --name google_apis_env --display-name "Python (google_apis_env)"
```

### 🐍 Option B. 표준 `pip` / `venv` 사용
```bash
python3 -m venv .venv
source .venv/bin/activate
pip install googlemaps requests pandas python-dotenv ipykernel
python -m ipykernel install --user --name google_apis_env --display-name "Python (google_apis_env)"
```

### 🔑 `.env` 파일 설정
프로젝트 루트의 `.env` 파일에 Google Cloud Console에서 발급받은 API 키를 설정합니다:
```env
GOOGLE_MAPS_API_KEY=AIzaSy...your_actual_api_key_here
```


In [ ]:
import os
import json
import datetime
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
import googlemaps

# 루트 디렉토리의 .env 파일을 자동으로 탐색하고 로드
env_path = find_dotenv()
load_dotenv(env_path, override=True)

# 깔끔한 JSON 출력을 위한 헬퍼 함수
def print_json(data, title=None, max_lines=35):
    if title:
        print(f"\n=== {title} ===")
    formatted = json.dumps(data, indent=2, ensure_ascii=False, default=str)
    lines = formatted.splitlines()
    if len(lines) > max_lines:
        print("\n".join(lines[:max_lines]))
        print(f"... [총 {len(lines)}줄 중 {max_lines}줄 출력됨 - 전체 데이터는 반환 객체 변수 참조]")
    else:
        print(formatted)

print(f"✅ 환경 설정 및 .env 로드 완료: {env_path if env_path else '기본 환경변수 사용'}")


## 🔑 1. API 키 로드 및 클라이언트 초기화


In [ ]:
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

if API_KEY:
    API_KEY = API_KEY.strip().strip('"').strip("'")

if not API_KEY or API_KEY == "YOUR_GOOGLE_MAPS_API_KEY_HERE":
    import getpass
    API_KEY = getpass.getpass("Google Maps API Key를 입력하세요: ").strip().strip('"').strip("'")

try:
    gmaps = googlemaps.Client(key=API_KEY)
    masked_key = f"{API_KEY[:6]}...{API_KEY[-4:]}" if len(API_KEY) > 10 else "***"
    print(f"✅ Google Maps Python SDK 클라이언트 초기화 성공 (키: {masked_key})")
except Exception as e:
    print("❌ 클라이언트 초기화 실패:", e)


## 📍 2. Geocoding API (주소 ➡️ 좌표 지오코딩)

### 📥 지원 입력 파라미터 (Input Attributes):
- `address`: 검색할 주소 문자열 (예: `"1600 Amphitheatre Pkwy, Mountain View, CA"`)
- `components`: 특정 국가/우편번호/행정구역으로 강제 필터링 (예: `{'country': 'US', 'postal_code': '94043'}`)
- `bounds`: 검색 바운딩 박스 가중치 (남서/북동 좌표 튜플)
- `region`: 국가 코드(ccTLD) 바이어스 (예: `"us"`, `"kr"`)
- `language`: 응답 주소 표기 언어 (예: `"ko"`, `"en"`)

### 📤 반환 출력 속성 (Output Attributes):
- `formatted_address`: 표준화된 전체 주소
- `place_id`: Google 고유 장소 식별자
- `types`: 장소/주소 유형 (`street_address`, `premise`, `locality` 등)
- `geometry`:
  - `location`: 위도(`lat`), 경도(`lng`)
  - `location_type`: 위치 정밀도 (`ROOFTOP`, `RANGE_INTERPOLATED`, `GEOMETRIC_CENTER`, `APPROXIMATE`)
  - `viewport` / `bounds`: 지도 뷰포트 표시 영역
- `address_components`: 도로명, 건물번호, 구/군, 시/도, 국가, 우편번호별 상세 분해 객체 배열

### 💡 실무 적용 시나리오:
1. **주소 정제 및 검증**: 사용자가 입력한 불완전한 주소를 표준 행정구역 체계로 변환 및 우편번호 검증
2. **국가 한정 검색(Component Restriction)**: 해외 주소 오매칭 방지를 위해 특정 국가(`country:KR`)로 한정 검색
3. **지도 뷰포트 자동 설정**: 반환된 `viewport` 영역을 통해 지도 라이브러리(Google Maps JS, Mapbox 등)의 적정 줌 레벨 자동 계산


In [ ]:
# 시나리오 1: 표준 지오코딩 및 주소 계층 컴포넌트 분석
target_address = "1600 Amphitheatre Parkway, Mountain View, CA 94043, USA"

try:
    geocode_result = gmaps.geocode(target_address, language="ko")
    print(f"✅ [시나리오 1] 주소 '{target_address}' 조회 결과 {len(geocode_result)}건:")
    
    if geocode_result:
        first = geocode_result[0]
        print(f"  - 표준 주소: {first.get('formatted_address')}")
        print(f"  - Place ID: {first.get('place_id')}")
        print(f"  - 위치 정밀도: {first.get('geometry', {}).get('location_type')}")
        loc = first.get('geometry', {}).get('location', {})
        print(f"  - 위도/경도: lat={loc.get('lat')}, lng={loc.get('lng')}")
        
        # 주소 컴포넌트 테이블
        df_comp = pd.DataFrame(first.get('address_components', []))
        display(df_comp)

    # 시나리오 2: Component Filtering (특정 국가/우편번호 강제 제한 검색)
    filtered_result = gmaps.geocode(
        "테헤란로 152",
        components={"country": "KR", "postal_code": "06236"},
        language="ko"
    )
    if filtered_result:
        print(f"\n✅ [시나리오 2] 컴포넌트 필터 적용 검색 (대한민국 06236 테헤란로 152):")
        print(f"  - 표준 주소: {filtered_result[0].get('formatted_address')}")
        print(f"  - 좌표: {filtered_result[0].get('geometry', {}).get('location')}")
        
except Exception as e:
    print("❌ Geocoding API 오류:", e)


## 🔄 3. Reverse Geocoding API (좌표 ➡️ 주소 역지오코딩)

### 📥 지원 입력 파라미터 (Input Attributes):
- `latlng`: 위도/경도 좌표 튜플 (예: `(37.50005, 127.0365)`)
- `result_type`: 특정 주소 유형으로 결과 필터링 (예: `['street_address']`, `['locality']`, `['postal_code']`)
- `location_type`: 위치 정밀도로 필터링 (예: `['ROOFTOP']`, `['APPROXIMATE']`)
- `language`: 결과 언어 (`"ko"`, `"en"`)

### 📤 반환 출력 속성 (Output Attributes):
- 정밀한 건물/지번 단위 주소부터 동/구/시/국가 단위까지 계층화된 주소 매칭 후보 목록 배열

### 💡 실무 적용 시나리오:
1. **모바일 GPS 기반 현재 위치 주소 표기**: 앱 사용자의 현재 GPS 좌표를 실시간 "도로명 주소"로 화면에 표기
2. **배달/택시 픽업 포인트 자동 지정**: `result_type=['street_address', 'premise']` 필터로 정확한 승하차 지점 주소 결정
3. **행정구역 자동 분류**: 좌표가 속한 시/도(`administrative_area_level_1`)를 판별하여 지역별 서비스 분기 처리


In [ ]:
# 서울 역삼동 GFC 좌표
coords = (37.50005, 127.0365)

try:
    # 1. 일반 역지오코딩 (전체 계층 반환)
    rev_all = gmaps.reverse_geocode(coords, language="ko")
    print(f"✅ 좌표 {coords} 일반 역지오코딩: 총 {len(rev_all)}개 계층 결과 반환\n")
    
    rev_records = []
    for idx, item in enumerate(rev_all[:6]):
        rev_records.append({
            "순번": idx + 1,
            "표준 주소": item.get("formatted_address"),
            "정밀도 (Location Type)": item.get("geometry", {}).get("location_type"),
            "주소 유형 (Types)": ", ".join(item.get("types", []))
        })
    df_rev = pd.DataFrame(rev_records)
    display(df_rev)

    # 2. 정밀 필터링: 정확한 건물/도로명(ROOFTOP & street_address)만 추출
    rev_filtered = gmaps.reverse_geocode(
        coords,
        result_type=["street_address", "premise"],
        location_type=["ROOFTOP"],
        language="ko"
    )
    if rev_filtered:
        print("🎯 [ROOFTOP 정밀 필터 결과]:", rev_filtered[0].get("formatted_address"))
except Exception as e:
    print("❌ Reverse Geocoding API 오류:", e)


## 🔍 4. Places API (New): 텍스트 검색 & 주변 시설 검색

최신 **Places API Modern v1** 엔드포인트를 사용합니다. 필요한 필드만 요청하는 **FieldMask**를 통해 성능과 비용을 최적화할 수 있습니다.

### 📥 지원 입력 파라미터 (Input Attributes):
- `textQuery`: 검색어 (자연어, 상호명, 카테고리 등)
- `includedType` / `includedTypes`: 장소 카테고리 (예: `["cafe", "restaurant", "hotel", "gas_station"]`)
- `locationRestriction` / `locationBias`: 검색 반경 (중심점 `center` + 반경 `radius` 미터, 또는 사각형 `rectangle`)
- `openNow`: 현재 영업 중인 장소만 필터링 (`true`/`false`)
- `minRating`: 최소 평점 필터 (예: `4.0`, `4.5`)
- `priceLevels`: 가격대 필터 (`PRICE_LEVEL_INEXPENSIVE`, `PRICE_LEVEL_MODERATE`, `PRICE_LEVEL_EXPENSIVE`)
- `rankPreference`: 정렬 기준 (`POPULARITY`(인기순), `DISTANCE`(거리순), `RELEVANCE`(관련도순))
- `maxResultCount`: 결과 개수 (1~20)
- `X-Goog-FieldMask`: 반환받을 필드 목록 (예: `places.id,places.displayName,places.rating,places.location`)

### 📤 반환 출력 속성 (Output Attributes):
- 장소 ID, 표시 이름, 주소, 평점, 총 리뷰수, 가격대, 영업상태, 카테고리, 좌표, 영업시간 등

### 💡 실무 적용 시나리오:
1. **조건부 장소 탐색**: "마운틴뷰 인근 평점 4.0 이상이며 지금 영업 중인 카페" 검색
2. **거리순 긴급 시설 탐색**: 사용자 현재 위치 기준 반경 2km 내 가장 가까운 주유소/전기차 충전소 거리순 정렬


In [ ]:
# 4.1 Places Text Search (New v1): 조건부 텍스트 검색 (평점/영업여부 필터)
url_text = "https://places.googleapis.com/v1/places:searchText"
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.userRatingCount,places.location,places.primaryType,places.businessStatus,places.regularOpeningHours"
}

body_text = {
    "textQuery": "Googleplex Mountain View",
    "minRating": 4.0,
    "languageCode": "ko"
}

sample_place_id = None
res_text = requests.post(url_text, headers=headers, json=body_text)

if res_text.status_code == 200:
    data = res_text.json()
    places = data.get("places", [])
    print(f"✅ Places 텍스트 검색 성공 ({len(places)}건 반환):")
    
    rows = []
    for p in places:
        rows.append({
            "장소명": p.get("displayName", {}).get("text"),
            "평점": p.get("rating"),
            "리뷰 수": p.get("userRatingCount"),
            "카테고리": p.get("primaryType"),
            "영업 상태": p.get("businessStatus"),
            "주소": p.get("formattedAddress"),
            "Place ID": p.get("id")
        })
    df_text = pd.DataFrame(rows)
    display(df_text)
    if places:
        sample_place_id = places[0].get("id")
else:
    print(f"❌ Text Search 오류 ({res_text.status_code}):", res_text.text)

# 4.2 Places Nearby Search (New v1): 반경 1,500m 내 카페 검색
url_nearby = "https://places.googleapis.com/v1/places:searchNearby"
headers_nearby = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.userRatingCount,places.primaryType,places.location"
}

body_nearby = {
    "includedTypes": ["cafe", "coffee_shop"],
    "maxResultCount": 5,
    "locationRestriction": {
        "circle": {
            "center": {"latitude": 37.422388, "longitude": -122.0841883},
            "radius": 1500.0
        }
    }
}

res_nearby = requests.post(url_nearby, headers=headers_nearby, json=body_nearby)
if res_nearby.status_code == 200:
    nearby_data = res_nearby.json()
    nearby_places = nearby_data.get("places", [])
    print(f"\n✅ Places 주변 1.5km 검색 성공 ({len(nearby_places)}개 카페 발견):")
    df_nearby = pd.DataFrame([
        {
            "장소명": p.get("displayName", {}).get("text"),
            "평점": p.get("rating"),
            "리뷰 수": p.get("userRatingCount"),
            "주소": p.get("formattedAddress"),
            "위치": p.get("location")
        } for p in nearby_places
    ])
    display(df_nearby)
else:
    print(f"❌ Nearby Search 오류 ({res_nearby.status_code}):", res_nearby.text)


## 🏢 5. Places API (New): 장소 상세 정보 전체 속성 조회 (`*` Wildcard FieldMask)

와일드카드 필드마스크 `X-Goog-FieldMask: *`를 지정하여 장소에 대해 Google이 보유한 **모든 속성(50여 개 이상의 필드)**을 완전히 추출합니다.

### 📥 지원 입력 파라미터 (Input Attributes):
- `place_id`: 고유 장소 식별자 (URL 경로 파라미터)
- `X-Goog-FieldMask`: `*` (전체 조회) 또는 개별 필드 쉼표 구분
- `languageCode`: 언어 설정

### 📤 제공되는 전체 속성 카테고리 (Output Attributes):
1. **식별 및 지오메트리**: `id`, `displayName`, `formattedAddress`, `location`, `viewport`, `plusCode`, `googleMapsUri`, `types`, `primaryType`
2. **연락처 및 웹**: `internationalPhoneNumber`, `nationalPhoneNumber`, `websiteUri`
3. **영업 시간**: `regularOpeningHours`, `currentOpeningHours`, `regularSecondaryOpeningHours` (드라이브스루, 해피아워 등)
4. **소개 및 리뷰**: `editorialSummary`, `rating`, `userRatingCount`, `priceLevel`, `reviews` (작성자, 평점, 본문, 작성일, 원문)
5. **다이닝 & 서비스 옵션**: `dineIn`, `delivery`, `takeout`, `curbsidePickup`, `reservable`, `servesBreakfast`, `servesLunch`, `servesDinner`, `servesBeer`, `servesWine`, `servesVegetarianFood`
6. **현대 시설 & 분위기**: `outdoorSeating`, `liveMusic`, `menuForChildren`, `goodForChildren`, `goodForGroups`, `restroom`, `allowsDogs`
7. **접근성 (Accessibility)**: `wheelchairAccessibleParking`, `wheelchairAccessibleEntrance`, `wheelchairAccessibleRestroom`, `wheelchairAccessibleSeating`
8. **주차 및 전기차(EV)**: `parkingOptions` (무료/유료 주차장, 발렛 등), `evChargeOptions` (충전 포트 수 및 커넥터 타입)
9. **결제 수단**: `paymentOptions` (신용카드, 체크카드, 현금전용, NFC 간편결제)
10. **사진(Photos)**: 사진 레퍼런스 및 해상도, 저작권 attribution

### 💡 실무 적용 시나리오:
1. **완전한 매장 상세 프로필 페이지 구축**: 영업시간, 메뉴 옵션, 고객 리뷰, 대표 사진을 하나의 API 호출로 렌더링
2. **배리어프리(무장애) 지도 앱**: 휠체어 탑승객을 위한 출입구/화장실/주차장 완비 매장 필터링
3. **반려동물 동반 가능(Pet-friendly) 및 EV 충전 식당 큐레이션**


In [ ]:
target_place_id = sample_place_id or "ChIJj61dQgK6j4AR4GeTYWZsKWw"

url_details = f"https://places.googleapis.com/v1/places/{target_place_id}"
headers_details = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "*"  # 모든 필드 일괄 요청
}

res_details = requests.get(url_details, headers=headers_details)

if res_details.status_code == 200:
    place_all = res_details.json()
    print(f"✅ 장소 상세 전체 페이로드 수신 성공! (제공된 최상위 필드 수: {len(place_all.keys())}개)\n")
    
    # 1. 기본 식별 및 연락처
    print("📌 [1. 기본 식별 및 연락처]")
    print(f"  - 장소명: {place_all.get('displayName', {}).get('text')}")
    print(f"  - Place ID: {place_all.get('id')}")
    print(f"  - 표준 주소: {place_all.get('formattedAddress')}")
    print(f"  - 전화번호: {place_all.get('internationalPhoneNumber', '제공안됨')}")
    print(f"  - 웹사이트: {place_all.get('websiteUri', '제공안됨')}")
    print(f"  - 소개 요약: {place_all.get('editorialSummary', {}).get('text', '정보 없음')}")
    
    # 2. 시설 편의성 및 옵션 분해
    print("\n🍽️ [2. 편의시설, 접근성 및 서비스 옵션]")
    feature_dict = {
        "매장 내 식사 (Dine In)": place_all.get("dineIn"),
        "배달 (Delivery)": place_all.get("delivery"),
        "포장 (Takeout)": place_all.get("takeout"),
        "예약 가능 (Reservable)": place_all.get("reservable"),
        "채식 메뉴 (Vegetarian)": place_all.get("servesVegetarianFood"),
        "야외 좌석 (Outdoor Seating)": place_all.get("outdoorSeating"),
        "반려동물 허용 (Allows Dogs)": place_all.get("allowsDogs"),
        "휠체어 접근성": place_all.get("accessibilityOptions"),
        "주차 옵션": place_all.get("parkingOptions"),
        "결제 수단": place_all.get("paymentOptions"),
        "전기차 충전 (EV)": place_all.get("evChargeOptions")
    }
    for k, v in feature_dict.items():
        if v is not None:
            print(f"  - {k}: {v}")

    # 3. 고객 리뷰 테이블
    reviews = place_all.get("reviews", [])
    print(f"\n💬 [3. 고객 리뷰 내역 ({len(reviews)}건)]")
    review_rows = []
    for r in reviews:
        review_rows.append({
            "작성자": r.get("authorAttribution", {}).get("displayName"),
            "평점": f"⭐ {r.get('rating')}",
            "작성일": r.get("relativePublishTimeDescription"),
            "리뷰": (r.get("text", {}).get("text", "")[:80] + "...") if len(r.get("text", {}).get("text", "")) > 80 else r.get("text", {}).get("text", "")
        })
    df_reviews = pd.DataFrame(review_rows)
    display(df_reviews)
    
    # 4. 전체 JSON 출력 (상위 30줄)
    print_json(place_all, title="Place Details 전체 원본 응답 (*)")
else:
    print(f"❌ Place Details 오류 ({res_details.status_code}):", res_details.text)


## ✍️ 6. Places Autocomplete API (실시간 자동완성 및 세션 토큰)

### 📥 지원 입력 파라미터 (Input Attributes):
- `input`: 사용자 실시간 입력 문자열 (예: `"Golden Gate"`)
- `sessionToken`: 자동완성 타이핑 ➡️ 최종 장소 선택까지 하나의 세션으로 묶어 과금을 최적화하는 UUID 토큰
- `origin`: 사용자의 현재 좌표 (자동완성 후보 장소까지의 **직선거리(distanceMeters)** 계산에 사용)
- `includedPrimaryTypes`: 특정 장소 유형만 추천 (예: `["tourist_attraction", "lodging"]`)
- `includedRegionCodes`: 추천 후보 국가 제한 (예: `["us", "kr"]`)
- `locationBias`: 검색 위치 바이어스 (원형 반경)

### 📤 반환 출력 속성 (Output Attributes):
- `suggestions`:
  - `placePrediction`:
    - `placeId`: 장소 ID
    - `text`: 전체 추천 텍스트
    - `structuredFormat`: 메인 텍스트(장소명) 및 서브 텍스트(주소/지역) 분리 구조체
    - `distanceMeters`: 기준 좌표(origin)로부터의 거리(미터)
    - `types`: 장소 유형 태그

### 💡 실무 적용 시나리오:
1. **검색창 인스턴트 자동완성 UI**: 키보드 입력 시 메인 장소명과 보조 주소를 분리 렌더링하고 사용자 위치로부터의 거리 표시
2. **Session Token 과금 절감**: 사용자가 글자를 칠 때마다 발생하는 Autocomplete 호출을 하나의 세션으로 묶어 Place Details 호출 시 1건의 비용으로 정산


In [ ]:
import uuid

# 세션 토큰 생성 (UUID v4)
session_token = str(uuid.uuid4())

url_auto = "https://places.googleapis.com/v1/places:autocomplete"
headers_auto = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY
}

body_auto = {
    "input": "Golden Gate",
    "sessionToken": session_token,
    "origin": {
        "latitude": 37.7749,
        "longitude": -122.4194  # 샌프란시스코 다운타운 기준
    },
    "includedPrimaryTypes": ["tourist_attraction", "park", "bridge", "establishment"],
    "languageCode": "ko"
}

res_auto = requests.post(url_auto, headers=headers_auto, json=body_auto)

if res_auto.status_code == 200:
    auto_data = res_auto.json()
    suggestions = auto_data.get("suggestions", [])
    print(f"✅ 자동완성 결과 {len(suggestions)}건 반환 (Session Token: {session_token[:8]}...):\n")
    
    sugg_list = []
    for s in suggestions:
        pred = s.get("placePrediction", {})
        dist_m = pred.get("distanceMeters")
        dist_km_str = f"{dist_m / 1000:.2f} km" if dist_m is not None else "계산불가"
        
        sugg_list.append({
            "메인 텍스트": pred.get("structuredFormat", {}).get("mainText", {}).get("text"),
            "서브 텍스트": pred.get("structuredFormat", {}).get("secondaryText", {}).get("text"),
            "기준점 거리": dist_km_str,
            "Place ID": pred.get("placeId"),
            "유형": ", ".join(pred.get("types", [])[:3])
        })
    df_sugg = pd.DataFrame(sugg_list)
    display(df_sugg)
else:
    print(f"❌ Autocomplete 오류 ({res_auto.status_code}):", res_auto.text)


## 🚗 7. Directions API (경로 탐색, 다중 경유지 최적화 & 교통 안내)

### 📥 지원 입력 파라미터 (Input Attributes):
- `origin` / `destination`: 출발지 및 목적지 (주소 문자열 또는 위경도 튜플)
- `mode`: 이동 수단 (`"driving"`, `"walking"`, `"bicycling"`, `"transit"`)
- `waypoints`: 중간 경유지 목록 (문자열 리스트 또는 좌표 리스트)
- `optimize_waypoints`: `True` 설정 시 외판원 문제(TSP)를 해결하여 경유지를 **최단 시간/거리 순서로 자동 재배열**
- `alternatives`: 대안 경로 탐색 여부 (`True`/`False`)
- `avoid`: 회피 옵션 (`["tolls"]`(유료도로), `["highways"]`(고속도로), `["ferries"]`(페리))
- `departure_time`: 출발 시각 (`"now"` 또는 미래 타임스탬프 ➡️ 실시간/예측 교통량 반영)
- `traffic_model`: 교통량 예측 모델 (`"best_guess"`, `"pessimistic"`, `"optimistic"`)
- `transit_mode`: 대중교통 세부 수단 (`["bus", "subway", "train", "tram", "rail"]`)
- `transit_routing_preference`: 대중교통 선호도 (`"less_walking"`, `"fewer_transfers"`)

### 📤 반환 출력 속성 (Output Attributes):
- `routes`:
  - `summary`: 주요 경유 도로명
  - `waypoint_order`: 최적화된 경유지 인덱스 방문 순서
  - `legs`: 각 구간별 거리(`distance`), 기본 소요시간(`duration`), 교통반영 소요시간(`duration_in_traffic`), 출발/도착 주소
  - `steps`: 턴바이턴 회전 안내(HTML 및 클린 텍스트), 동작(`maneuver`), 구간 거리/시간
  - `overview_polyline`: 지도에 경로 선을 그리기 위한 인코딩된 폴리라인 문자열

### 💡 실무 적용 시나리오:
1. **물류/배송 차량 다중 경유지 순서 최적화 (`optimize_waypoints=True`)**: 5개 배송 목적지를 입력받아 최적의 방문 순서와 총 운행 시간 도출
2. **출퇴근 시간대 교통 혼잡 예측 비교**: `traffic_model="pessimistic"` vs `"optimistic"`을 비교하여 최악의 지연 시간 산출
3. **내비게이션 턴바이턴 주행 가이드**: 단계별 회전 및 주행 안내 테이블 렌더링


In [ ]:
# 시나리오 1: 다중 경유지 방문 순서 최적화 (TSP Route Optimization)
origin = "San Francisco, CA"
destination = "San Jose, CA"
waypoints = [
    "Stanford University, CA",
    "San Mateo, CA",
    "Palo Alto, CA"
]

try:
    # optimize_waypoints=True 설정 시 경유지를 최단 순서로 정렬
    opt_directions = gmaps.directions(
        origin=origin,
        destination=destination,
        waypoints=waypoints,
        optimize_waypoints=True,
        mode="driving",
        departure_time="now",
        traffic_model="best_guess"
    )

    if opt_directions:
        route = opt_directions[0]
        print(f"✅ 다중 경유지 경로 최적화 완료 (주요 도로: {route.get('summary')}):")
        print(f"🎯 최적화된 경유지 방문 순서 인덱스: {route.get('waypoint_order')}")
        
        # 각 구간(Leg)별 거리 및 소요시간 요약
        leg_rows = []
        for i, leg in enumerate(route.get("legs", [])):
            leg_rows.append({
                "구간": f"구간 {i + 1}",
                "출발지": leg.get("start_address")[:30] + "...",
                "도착지": leg.get("end_address")[:30] + "...",
                "구간 거리": leg.get("distance", {}).get("text"),
                "기본 소요시간": leg.get("duration", {}).get("text"),
                "실시간 소요시간": leg.get("duration_in_traffic", {}).get("text", "N/A")
            })
        df_legs = pd.DataFrame(leg_rows)
        display(df_legs)

        # 추천 경로의 첫 5개 턴바이턴 주행 안내
        steps = route["legs"][0]["steps"]
        print(f"\n📋 [첫 번째 구간의 단계별 턴바이턴 가이드 (상위 5단계)]:")
        import re
        step_rows = []
        for s_idx, s in enumerate(steps[:5]):
            step_rows.append({
                "단계": s_idx + 1,
                "주행 안내": re.sub('<[^<]+?>', '', s.get("html_instructions", "")),
                "거리": s.get("distance", {}).get("text"),
                "시간": s.get("duration", {}).get("text"),
                "동작": s.get("maneuver", "직진")
            })
        display(pd.DataFrame(step_rows))

except Exception as e:
    print("❌ Directions API 오류:", e)


## 📏 8. Distance Matrix API (다중 출발지-도착지 거리 행렬)

### 📥 지원 입력 파라미터 (Input Attributes):
- `origins`: 출발지 목록 (주소 문자열 리스트 또는 좌표 리스트)
- `destinations`: 목적지 목록
- `mode`: 이동 수단 (`"driving"`, `"walking"`, `"bicycling"`, `"transit"`)
- `departure_time`: 실시간/미래 출발 시간 (교통량 반영)
- `traffic_model`: `"best_guess"`, `"pessimistic"`, `"optimistic"`
- `avoid`: `["tolls"]`, `["highways"]`, `["ferries"]`

### 📤 반환 출력 속성 (Output Attributes):
- `origin_addresses` / `destination_addresses`: 정규화된 출발지/도착지 주소 배열
- `rows[i].elements[j]`: $i$번째 출발지와 $j$번째 목적지 간의:
  - `status`: `"OK"`, `"ZERO_RESULTS"`
  - `distance`: 미터(`value`) 및 포맷된 문자열(`text`)
  - `duration`: 초(`value`) 및 포맷된 문자열(`text`)
  - `duration_in_traffic`: 실시간 교통 반영 소요시간

### 💡 실무 적용 시나리오:
1. **라이드헤일링/배달 기사 최적 매칭**: 3명의 배달원 위치와 2명의 고객 주문지 간 N x M 행렬을 계산하여 가장 빨리 도착할 수 있는 기사 배차
2. **물류 거점 허브 선정**: 복수의 물류센터 후보지와 복수의 대리점 간 총 이동 시간/비용 비교 분석


In [ ]:
origins = ["San Francisco, CA", "Oakland, CA", "San Jose, CA"]
destinations = ["Mountain View, CA", "Palo Alto, CA"]

try:
    matrix_result = gmaps.distance_matrix(
        origins=origins,
        destinations=destinations,
        mode="driving",
        departure_time="now",
        traffic_model="best_guess"
    )

    matrix_rows = []
    for i, origin_name in enumerate(matrix_result.get("origin_addresses", [])):
        row_elements = matrix_result["rows"][i]["elements"]
        for j, dest_name in enumerate(matrix_result.get("destination_addresses", [])):
            element = row_elements[j]
            if element.get("status") == "OK":
                matrix_rows.append({
                    "출발지 (기사/거점)": origin_name,
                    "목적지 (고객/도착점)": dest_name,
                    "거리": element.get("distance", {}).get("text"),
                    "표준 소요시간": element.get("duration", {}).get("text"),
                    "실시간 소요시간 (교통 반영)": element.get("duration_in_traffic", {}).get("text", "정보 없음")
                })

    df_matrix = pd.DataFrame(matrix_rows)
    print(f"✅ {len(origins)}개 출발지 x {len(destinations)}개 목적지 거리 행렬 계산 완료:")
    display(df_matrix)
except Exception as e:
    print("⚠️ Distance Matrix API 참고:", e)
    print("💡 콘솔 활성화 링크: https://console.cloud.google.com/apis/library/distancematrix-backend.googleapis.com")


## ⛰️ 9. Elevation API (고도 측정 & 경로 고도 프로파일)

### 📥 지원 입력 파라미터 (Input Attributes):
- `locations`: 단일 좌표 또는 복수 좌표 리스트 (특정 지점 고도 측정)
- `path` & `samples`: 출발지에서 목적지까지의 경로 좌표 목록과 샘플링 포인트 개수 (경로를 따른 연속 고도 프로파일 생성)

### 📤 반환 출력 속성 (Output Attributes):
- `elevation`: 해발 고도 (미터 단위 부동소수점)
- `location`: 해당 지점 위도/경도
- `resolution`: 고도 데이터의 보간 해상도 (미터 단위)

### 💡 실무 적용 시나리오:
1. **등산/사이클링 경로 고도 획득량(Elevation Gain) 및 경사도 계산**: 경로상 10개 지점을 샘플링하여 오르막/내리막 고도 프로파일 차트 생성
2. **드론/UAV 자율 비행 지형 충돌 방지**: 비행 경로상의 지형 최고 고도를 사전 확인하여 안전 비행 고도 산정


In [ ]:
# 1. 특정 주요 랜드마크 고도 비교
landmarks = [
    {"name": "에베레스트 정상 (세계 최고봉)", "coords": (27.9881, 86.9250)},
    {"name": "데스밸리 배드워터 (북미 최저점)", "coords": (36.2503, -116.8258)},
    {"name": "한라산 백록담", "coords": (33.3617, 126.5332)},
    {"name": "구글 본사 (마운틴뷰)", "coords": (37.4220, -122.0841)}
]

try:
    coords_list = [l["coords"] for l in landmarks]
    elevation_results = gmaps.elevation(coords_list)
    
    elevation_records = []
    for l, res in zip(landmarks, elevation_results):
        elev_m = res.get("elevation", 0)
        elevation_records.append({
            "위치 명칭": l["name"],
            "위도": res.get("location", {}).get("lat"),
            "경도": res.get("location", {}).get("lng"),
            "해발 고도 (미터)": f"{elev_m:.2f} m",
            "해발 고도 (피트)": f"{elev_m * 3.28084:.2f} ft",
            "측정 해상도": f"{res.get('resolution', 0):.2f} m"
        })
    df_elevation = pd.DataFrame(elevation_records)
    print("✅ 랜드마크별 해발 고도 측정 결과:")
    display(df_elevation)

    # 2. 경로 고도 프로파일 샘플링 (San Francisco -> Mountain View 간 5개 샘플 지점)
    path_sample = [(37.7749, -122.4194), (37.4220, -122.0841)]
    path_elev = gmaps.elevation_along_path(path_sample, samples=5)
    print(f"\n✅ 경로 고도 프로파일 샘플링 ({len(path_elev)}개 지점 수신):")
    df_path = pd.DataFrame([
        {
            "샘플 지점": f"Point #{idx + 1}",
            "좌표": f"({p['location']['lat']:.4f}, {p['location']['lng']:.4f})",
            "고도 (m)": f"{p['elevation']:.2f} m"
        } for idx, p in enumerate(path_elev)
    ])
    display(df_path)

except Exception as e:
    print("⚠️ Elevation API 참고:", e)
    print("💡 콘솔 활성화 링크: https://console.cloud.google.com/apis/library/elevation-backend.googleapis.com")


## ⏰ 10. Time Zone API (시간대 및 서머타임 DST 조회)

### 📥 지원 입력 파라미터 (Input Attributes):
- `location`: 대상 지점 위도/경도 좌표
- `timestamp`: UTC 기준 타임스탬프 (계절별 서머타임 적용 여부를 정확히 계산하기 위해 필수)
- `language`: 타임존 이름 언어 표기

### 📤 반환 출력 속성 (Output Attributes):
- `timeZoneId`: IANA 표준 타임존 ID (예: `"America/New_York"`, `"Asia/Seoul"`)
- `timeZoneName`: 현지 타임존 명칭 (예: `"동부 표준시"`, `"한국 표준시"`)
- `rawOffset`: 표준 UTC 오프셋 (초 단위)
- `dstOffset`: 서머타임(Daylight Saving Time) 추가 오프셋 (초 단위)
- `status`: `"OK"`

### 💡 실무 적용 시나리오:
1. **글로벌 예약 시스템 현지 시간 계산**: 항공권, 호텔, 렌터카 예약 시 목적지 좌표 기반 현지 도착 시간 및 서머타임 자동 적용
2. **IoT 기기 GPS 연동 시간 동기화**: 글로벌 배포된 장비가 GPS 좌표를 획득하면 Time Zone API로 현지 시간 자동 세팅


In [ ]:
cities = [
    {"city": "대한민국 서울", "coords": (37.5665, 126.9780)},
    {"city": "미국 뉴욕 (서머타임 검증)", "coords": (40.7128, -74.0060)},
    {"city": "영국 런던", "coords": (51.5074, -0.1278)},
    {"city": "호주 시드니 (남반구)", "coords": (-33.8688, 151.2093)}
]

now_timestamp = datetime.datetime.now(datetime.timezone.utc).timestamp()

try:
    timezone_records = []
    for c in cities:
        tz_res = gmaps.timezone(location=c["coords"], timestamp=now_timestamp, language="ko")
        if tz_res.get("status") == "OK":
            raw_h = tz_res.get("rawOffset", 0) / 3600
            dst_h = tz_res.get("dstOffset", 0) / 3600
            total_h = raw_h + dst_h
            timezone_records.append({
                "도시명": c["city"],
                "타임존 ID": tz_res.get("timeZoneId"),
                "타임존 명칭": tz_res.get("timeZoneName"),
                "표준 오프셋": f"{raw_h:+.1f} 시간",
                "서머타임 (DST)": f"{dst_h:+.1f} 시간" if dst_h != 0 else "미적용 (0h)",
                "최종 UTC 오프셋": f"UTC{total_h:+.1f}"
            })
    df_tz = pd.DataFrame(timezone_records)
    print("✅ 글로벌 주요 도시 타임존 및 서머타임 조회 결과:")
    display(df_tz)
except Exception as e:
    print("⚠️ Time Zone API 참고:", e)
    print("💡 콘솔 활성화 링크: https://console.cloud.google.com/apis/library/timezone-backend.googleapis.com")


## 📶 11. Geolocation API (네트워크/기지국/IP 기반 위치 추정)

### 📥 지원 입력 파라미터 (Input Attributes):
- `considerIp`: IP 주소 기반 위치 추정 포함 여부 (`true`/`false`)
- `radioType`: 무선 통신 규격 (`"lte"`, `"gsm"`, `"cdma"`, `"wcdma"`, `"nr"` 5G)
- `carrier`: 통신사 명칭
- `cellTowers`: 주변 기지국 정보 배열 (`cellId`, `locationAreaCode`, `signalStrength` 신호 세기 등)
- `wifiAccessPoints`: 주변 Wi-Fi AP 목록 (`macAddress` BSSID, `signalStrength` RSSI, `channel`, `signalToNoiseRatio`)

### 📤 반환 출력 속성 (Output Attributes):
- `location`: 추정 위도(`lat`), 경도(`lng`)
- `accuracy`: 위치 정확도 오차 반경 (미터 단위)

### 💡 실무 적용 시나리오:
1. **실내/지하 GPS 음영지역 측위**: GPS 신호 수신이 불가능한 쇼핑몰, 지하철, 빌딩 내부에서 주변 Wi-Fi AP 스캔 정보로 기기 위치 특정
2. **저전력 IoT 자산 추적**: 고가의 GPS 모듈 없이 Wi-Fi BSSID 스캔 데이터만으로 물류 컨테이너/자산 위치 추적
3. **보안 및 이상 로그인 감지**: IP/네트워크 신호 기반 위치와 평소 접속 위치의 불일치 탐지


In [ ]:
try:
    # IP 및 네트워크 신호 기반 기기 위치 추정
    geolocate_res = gmaps.geolocate(consider_ip=True)
    print("✅ Geolocation 기기 위치 추정 성공:")
    print(f"  - 추정 좌표: {geolocate_res.get('location')}")
    print(f"  - 정확도 반경: {geolocate_res.get('accuracy')} 미터")
    print_json(geolocate_res, title="Geolocation API 응답 JSON")
except Exception as e:
    print("⚠️ Geolocation API 참고:", e)
    print("💡 콘솔 활성화 링크: https://console.cloud.google.com/apis/library/geolocation.googleapis.com")


## 🛣️ 12. Roads API (도로 스냅 Snap to Roads & 제한속도)

### 📥 지원 입력 파라미터 (Input Attributes):
- `path`: 차량 주행 중 수집된 연속된 GPS 위경도 좌표 리스트 (예: `[(lat1, lng1), (lat2, lng2), ...]`)
- `interpolate`: GPS 수집 간격이 넓은 구간을 실제 도로 형상에 맞춰 보간할지 여부 (`True`/`False`)
- `points`: 가장 가까운 도로를 찾기 위한 좌표 목록

### 📤 반환 출력 속성 (Output Attributes):
- `snappedPoints`:
  - `location`: 실제 도로 중심선에 스냅된 보정 위도/경도
  - `originalIndex`: 원본 입력 좌표와의 매핑 인덱스
  - `placeId`: 스냅된 도로 세그먼트의 고유 Place ID

### 💡 실무 적용 시나리오:
1. **차량 관제(FMS) GPS 궤적 보정**: 빌딩 숲, 터널 주변에서 튀는 GPS 궤적을 실제 도로 네트워크 위로 매끄럽게 보정
2. **운행 속도 제한(Speed Limit) 준수율 분석**: 도로 세그먼트의 법정 제한속도와 차량 실시간 속도를 비교하여 안전운행 점수 산출


In [ ]:
# 주행 중 수집된 약간의 오차가 있는 GPS 궤적 예시 (샌프란시스코 인근)
sample_gps_path = [
    (37.7749, -122.4194),
    (37.7752, -122.4178),
    (37.7758, -122.4162),
    (37.7765, -122.4145)
]

try:
    # 도로 형상에 맞춰 보간(interpolate=True)하여 스냅
    snapped = gmaps.snap_to_roads(sample_gps_path, interpolate=True)
    print(f"✅ Roads API 도로 스냅 완료 (원본 {len(sample_gps_path)}개 지점 ➡️ 보정 {len(snapped)}개 지점):\n")
    
    snap_records = []
    for idx, pt in enumerate(snapped):
        snap_records.append({
            "포인트": f"Snap #{idx + 1}",
            "보정 위도": pt.get("location", {}).get("latitude"),
            "보정 경도": pt.get("location", {}).get("longitude"),
            "원본 매핑 인덱스": pt.get("originalIndex", "보간 지점(Interpolated)"),
            "도로 Place ID": pt.get("placeId")
        })
    df_snapped = pd.DataFrame(snap_records)
    display(df_snapped)
except Exception as e:
    print("⚠️ Roads API 참고:", e)
    print("💡 콘솔 활성화 링크: https://console.cloud.google.com/apis/library/roads.googleapis.com")


## 📊 13. 종합 속성 매트릭스 & 실무 활용 레퍼런스

| API 서비스 | 주요 입력 파라미터 (Inputs) | 핵심 출력 속성 (Outputs) | 대표 실무 활용 시나리오 |
| :--- | :--- | :--- | :--- |
| **Geocoding** | `address`, `components`, `bounds`, `region`, `language` | `formatted_address`, `geometry(location, type, viewport)`, `address_components`, `place_id` | 주소 정규화 및 유효성 검증, 국가 제한 주소 검색, 지도 초기 영역 자동 피팅 |
| **Reverse Geocoding** | `latlng`, `result_type`, `location_type`, `language` | 계층형 주소 목록 (`street_address` ~ `country`), `types`, `place_id` | GPS 좌표의 도로명 변환, 택시/배달 픽업 위치 자동 설정, 행정구역 태깅 |
| **Places (New) Search** | `textQuery`, `includedTypes`, `locationRestriction`, `openNow`, `minRating`, `rankPreference` | 장소 목록, `displayName`, `rating`, `userRatingCount`, `regularOpeningHours`, `location` | "영업중 + 평점 4.5+" 조건부 맛집 추천, 반경 기반 주변 시설 거리순/인기순 탐색 |
| **Places (New) Details** | `placeId`, `X-Goog-FieldMask: *`, `languageCode` | 50+ 속성 (`reviews`, `photos`, `editorialSummary`, `accessibility`, `parking`, `evCharge`, `dineIn`) | 포털/앱 내 완전한 장소 상세 화면 구축, 휠체어/반려동물/전기차 친화 시설 필터링 |
| **Places Autocomplete** | `input`, `sessionToken`, `origin`, `includedPrimaryTypes`, `locationBias` | `suggestions(mainText, secondaryText, distanceMeters, placeId)` | 검색창 실시간 자동완성, 내 위치 기준 거리 표기, 세션 토큰을 통한 과금 절감 |
| **Directions** | `origin`, `destination`, `mode`, `waypoints(optimize=True)`, `traffic_model`, `avoid` | `legs(distance, duration, traffic)`, `steps(turn-by-turn)`, `waypoint_order`, `polyline` | 다중 경유지 최적 순서 배송(TSP), 실시간 교통 혼잡 예측 ETA, 단계별 내비게이션 |
| **Distance Matrix** | `origins[]`, `destinations[]`, `mode`, `departure_time`, `traffic_model` | N x M 요소 행렬 (`distance`, `duration`, `duration_in_traffic`, `status`) | 배달/택시 기사-고객 최적 배차, 복수 거점 간 물류 이동 비용 및 시간 산출 |
| **Elevation** | `locations[]`, `path[]`, `samples` | `elevation`(해발 고도 m), `resolution`, 고도 프로파일 | 등산/자전거 코스 경사도 및 획득고도 분석, 드론 지형 안전고도 시뮬레이션 |
| **Time Zone** | `location`, `timestamp`, `language` | `timeZoneId`, `timeZoneName`, `rawOffset`, `dstOffset` (서머타임) | 글로벌 항공/호텔 예약 현지 시간 변환, IoT GPS 시간 자동 동기화 |
| **Geolocation** | `considerIp`, `radioType`, `cellTowers[]`, `wifiAccessPoints[]` | `location(lat, lng)`, `accuracy`(오차 반경 m) | 지하/실내 GPS 음영지역 측위, Wi-Fi BSSID 기반 자산 추적, 부정 로그인 탐지 |
| **Roads** | `path[]`, `interpolate`, `points[]` | `snappedPoints(location, originalIndex, placeId)` | 차량 GPS 궤적 도로 스냅 보정, 구간별 도로 법정 속도 준수율 모니터링 |

---

### 📚 공식 개발자 문서 & 레퍼런스
- 🌐 [Google Maps Platform 공식 문서](https://developers.google.com/maps/documentation)
- 🏢 [Places API (New) Web Service 개요](https://developers.google.com/maps/documentation/places/web-service/op-overview)
- 🚗 [Directions API 개발자 가이드](https://developers.google.com/maps/documentation/directions/overview)
- 🐍 [google-maps-services-python GitHub](https://github.com/googlemaps/google-maps-services-python)
- ⚙️ [Google Cloud Console API 라이브러리](https://console.cloud.google.com/apis/library)
